# A Computational Framework for Comparative Analysis of Free Trade Agreements
**Diyouva Christa Novith** | Machine Learning Foundation with Python | Spring 2026

---
This notebook walks through all stages of the pipeline:
1. PDF Extraction & Clause Segmentation
2. Embeddings & Vector Database
3. LLM Classification (LLaMA 3.3 70B vs Qwen 3 32B, zero-shot → few-shot → chain-of-thought)
4. Cross-Agreement Comparison
5. Results Visualisation & Findings

**Free models used — no payment required:**
| Model | Provider | Free Key |
|-------|----------|----------|
| LLaMA 3.3 70B | Groq | https://console.groq.com |
| Qwen 3 32B | Groq (Alibaba Cloud) | https://console.groq.com |

In [ ]:
import sys, os
sys.path.insert(0, '..')

# ── Paste your FREE API key here ──────────────────────────────────────────────
# Groq  (LLaMA 3.3 + Qwen 3) → free key at https://console.groq.com
os.environ['GROQ_API_KEY'] = ''  # ← paste here

import json
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path
from IPython.display import Markdown, display

from config import RESULT_DIR, RAW_DIR, POLICY_CATEGORIES, AGREEMENTS

plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.family'] = 'DejaVu Sans'
print('Setup complete.')

---
## Step 1 – PDF Extraction & Clause Segmentation

In [ ]:
# Skip if already extracted — data/raw/all_provisions.json exists
if not (RAW_DIR / 'all_provisions.json').exists():
    from src.extraction import run_extraction
    provisions = run_extraction()
else:
    print('Provisions already extracted. Loading from file…')

In [ ]:
# Load & inspect extracted provisions
with open(RAW_DIR / 'all_provisions.json') as f:
    provisions = json.load(f)

df = pd.DataFrame(provisions)
print(f'Total provisions: {len(df):,}')
display(df.groupby(['agreement', 'doc_type']).size().reset_index(name='count'))

In [ ]:
# Provision count by agreement
fig, ax = plt.subplots(figsize=(8, 4))
counts = df.groupby('agreement').size()
colors = ['#2166AC', '#D6604D', '#4DAC26']
counts.plot(kind='bar', ax=ax, color=colors, edgecolor='white')
ax.set_title('Provisions Extracted per Agreement', fontsize=13, fontweight='bold')
ax.set_xlabel('')
ax.set_ylabel('Number of Provisions')
ax.tick_params(axis='x', rotation=0)
for p in ax.patches:
    ax.annotate(f'{int(p.get_height()):,}', (p.get_x() + p.get_width()/2, p.get_height()),
                ha='center', va='bottom', fontsize=10)
plt.tight_layout()
plt.savefig(RESULT_DIR / 'fig_provision_counts.png', bbox_inches='tight')
plt.show()

In [ ]:
# Sample provisions
display(df[['agreement', 'doc_type', 'article', 'text']].sample(5))

---
## Step 2 – Embeddings & Vector Database

In [ ]:
from src.embedding import build_vector_store, retrieve_similar

# Skip if already built
import chromadb
from config import CHROMA_DIR, CHROMA_COLLECTION
client = chromadb.PersistentClient(path=str(CHROMA_DIR))
existing = [c.name for c in client.list_collections()]

if CHROMA_COLLECTION not in existing:
    build_vector_store(provisions)
else:
    print(f'Vector store already exists ({CHROMA_COLLECTION}). Skipping rebuild.')

In [ ]:
# Test semantic retrieval
query = 'rules of origin criteria for goods'
results = retrieve_similar(query, n_results=6)
print(f'Query: "{query}"\n')
for r in results:
    print(f"[{r['agreement']} | dist={r['distance']:.3f}] {r['article']}")
    print(f"  {r['text'][:200]}…\n")

In [ ]:
# Retrieve same topic from each agreement for comparison
topic = 'tariff reduction schedule'
print(f'Retrieving: "{topic}" from each agreement\n')
for agreement in AGREEMENTS:
    r = retrieve_similar(topic, agreement_filter=agreement, n_results=1)
    if r:
        print(f'── {agreement} ──')
        print(f"  Article: {r[0]['article']}")
        print(f"  {r[0]['text'][:300]}\n")

---
## Step 3 – LLM Classification

We compare two **free** models across three prompt strategies:

| | Zero-Shot | Few-Shot | Chain-of-Thought |
|---|---|---|---|
| **LLaMA 3.3 70B** (Groq) | ✅ | ✅ | ✅ |
| **Qwen 3 32B** (Groq) | ✅ | ✅ | ✅ |

### 3a. Zero-Shot — LLaMA 3.3 70B (Groq)

In [ ]:
from src.classification import classify_provisions

results_llama_zs = classify_provisions(provisions, model='llama', strategy='zero_shot', limit=200)
df_llama_zs = pd.DataFrame(results_llama_zs)
print(df_llama_zs['category'].value_counts())

### 3b. Few-Shot — LLaMA 3.3 70B (Groq)

In [ ]:
results_llama_fs = classify_provisions(provisions, model='llama', strategy='few_shot', limit=200)
df_llama_fs = pd.DataFrame(results_llama_fs)

### 3c. Chain-of-Thought — LLaMA 3.3 70B (Groq)

In [ ]:
results_llama_cot = classify_provisions(provisions, model='llama', strategy='cot', limit=200)
df_llama_cot = pd.DataFrame(results_llama_cot)

### 3d. Zero-Shot — Qwen 3 32B (Groq)

In [ ]:
results_qwen_zs = classify_provisions(provisions, model='qwen', strategy='zero_shot', limit=200)
df_qwen_zs = pd.DataFrame(results_qwen_zs)

In [ ]:
# Compare category distributions: LLaMA vs Qwen, across strategies
strategy_labels = [
    'Zero-Shot\n(LLaMA 3.3)',
    'Few-Shot\n(LLaMA 3.3)',
    'CoT\n(LLaMA 3.3)',
    'Zero-Shot\n(Qwen 3)',
]
dfs = [df_llama_zs, df_llama_fs, df_llama_cot, df_qwen_zs]

cats_to_show = [c for c in POLICY_CATEGORIES if c != 'Other']
comp_matrix = pd.DataFrame(
    {label: df['category'].value_counts().reindex(cats_to_show, fill_value=0)
     for label, df in zip(strategy_labels, dfs)}
)

fig, ax = plt.subplots(figsize=(13, 5))
comp_matrix.T.plot(kind='bar', ax=ax, colormap='tab10', edgecolor='white', width=0.8)
ax.set_title('Category Distribution by Model & Prompt Strategy (n=200 each)', fontsize=12, fontweight='bold')
ax.set_xlabel('Model / Strategy')
ax.set_ylabel('Provision Count')
ax.tick_params(axis='x', rotation=0)
ax.legend(loc='upper right', fontsize=7, ncol=2, bbox_to_anchor=(1.18, 1))
plt.tight_layout()
plt.savefig(RESULT_DIR / 'fig_strategy_comparison.png', bbox_inches='tight')
plt.show()
print(comp_matrix)

In [ ]:
# Agreement consistency: LLaMA Zero-Shot vs CoT
ids = df_llama_zs['id'].tolist()
cat_zs  = dict(zip(df_llama_zs['id'], df_llama_zs['category']))
cat_cot = dict(zip(df_llama_cot['id'], df_llama_cot['category']))
shared  = [i for i in ids if i in cat_cot]
agree   = sum(cat_zs[i] == cat_cot[i] for i in shared)
print(f'LLaMA 3.3 — Zero-shot vs CoT agreement: {agree}/{len(shared)} ({100*agree/len(shared):.1f}%)')

# LLaMA vs Qwen consistency (zero-shot)
cat_qwen = dict(zip(df_qwen_zs['id'], df_qwen_zs['category']))
shared2 = [i for i in ids if i in cat_qwen]
agree2  = sum(cat_zs[i] == cat_qwen[i] for i in shared2)
print(f'LLaMA 3.3 vs Qwen 3 — Zero-shot agreement: {agree2}/{len(shared2)} ({100*agree2/len(shared2):.1f}%)')

In [ ]:
# Manual validation sample — review these 10 provisions against source text
sample_valid = df_llama_cot.sample(10)[['agreement', 'article', 'text', 'category']]
sample_valid = sample_valid.copy()
sample_valid['text'] = sample_valid['text'].str[:200]
display(sample_valid)

---
## Step 4 – Cross-Agreement Comparison
Using **LLaMA 3.3 70B** (Groq) for the LLM-generated comparison narratives.
See Part 5 for Qwen-based results loaded from the completed classification runs.

In [ ]:
from src.comparison import compare_category, build_comparison_matrix

KEY_CATS = [
    'Rules of Origin',
    'Tariff Commitments',
    'Non-Tariff Measures',
    'Dispute Settlement',
    'Trade in Services',
]

comparisons = {}
for cat in KEY_CATS:
    print(f'\n── Comparing: {cat} ──')
    result = compare_category(cat, model='llama')
    comparisons[cat] = result
    print(result['analysis'][:500] + '…')

In [ ]:
# Print full analysis for one category
cat = 'Rules of Origin'
display(Markdown(f'## {cat}\n\n' + comparisons[cat]['analysis']))

In [ ]:
# Category × Agreement matrix (heatmap)
matrix = build_comparison_matrix(RESULT_DIR / 'classified_llama_zero_shot.json')
df_matrix = pd.DataFrame(matrix).T.fillna(0).astype(int)

fig, ax = plt.subplots(figsize=(10, 6))
im = ax.imshow(df_matrix.values, cmap='Blues', aspect='auto')
ax.set_xticks(range(len(df_matrix.columns)))
ax.set_xticklabels(df_matrix.columns, fontsize=11, fontweight='bold')
ax.set_yticks(range(len(df_matrix.index)))
ax.set_yticklabels(df_matrix.index, fontsize=9)
for i in range(len(df_matrix.index)):
    for j in range(len(df_matrix.columns)):
        val = df_matrix.values[i, j]
        ax.text(j, i, str(val), ha='center', va='center',
                color='white' if val > df_matrix.values.max()*0.6 else 'black', fontsize=9)
ax.set_title('Provision Count by Category × Agreement\n(LLaMA 3 Zero-Shot Classification)', fontsize=12, fontweight='bold')
plt.colorbar(im, ax=ax, label='Provision Count')
plt.tight_layout()
plt.savefig(RESULT_DIR / 'fig_category_matrix.png', bbox_inches='tight')
plt.show()

---
## Step 5 – Findings & Policy Implications

In [ ]:
# Category distribution per agreement (stacked bar)
with open(RESULT_DIR / 'classified_llama_zero_shot.json') as f:
    clf = json.load(f)
df_clf = pd.DataFrame(clf)

pivot = df_clf.groupby(['agreement', 'category']).size().unstack(fill_value=0)
pivot = pivot.reindex(columns=POLICY_CATEGORIES, fill_value=0)

ax = pivot.plot(kind='bar', stacked=True, figsize=(10, 5), colormap='tab20', edgecolor='none')
ax.set_title('Policy Category Distribution per Agreement\n(LLaMA 3 Zero-Shot)', fontsize=12, fontweight='bold')
ax.set_xlabel('')
ax.set_ylabel('Provision Count')
ax.tick_params(axis='x', rotation=0)
ax.legend(loc='upper right', fontsize=7, bbox_to_anchor=(1.35, 1), ncol=1)
plt.tight_layout()
plt.savefig(RESULT_DIR / 'fig_category_distribution.png', bbox_inches='tight')
plt.show()

In [ ]:
# Print all comparison analyses as Markdown
for cat, result in comparisons.items():
    display(Markdown(f'---\n## {cat}\n\n{result["analysis"]}'))

In [ ]:
# Export summary table
summary = []
for cat, result in comparisons.items():
    provs = result['provisions_used']
    summary.append({
        'Category': cat,
        'RCEP provisions': len(provs.get('RCEP', [])),
        'AHKFTA provisions': len(provs.get('AHKFTA', [])),
        'AANZFTA provisions': len(provs.get('AANZFTA', [])),
        'Analysis (excerpt)': result['analysis'][:200] + '…',
    })
df_summary = pd.DataFrame(summary)
df_summary.to_csv(RESULT_DIR / 'comparison_summary.csv', index=False)
display(df_summary[['Category', 'RCEP provisions', 'AHKFTA provisions', 'AANZFTA provisions']])

In [ ]:
print('\n✅ Analysis complete!')
print('Results saved to data/results/')
print('Figures: fig_provision_counts.png, fig_strategy_comparison.png,')
print('         fig_category_matrix.png, fig_category_distribution.png')

---
# 📊 Part 5 — Results & Analysis
This section loads the classified JSONs we actually produced (LLaMA + Qwen) and generates the figures for the final report.

In [ ]:
import json, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from pathlib import Path
from collections import Counter
sns.set_theme(style='whitegrid', context='talk')

RESULT_DIR = Path('../data/results')

runs = {}
for name in ['llama_zero_shot', 'llama_few_shot', 'llama_cot',
             'qwen_zero_shot', 'qwen_few_shot', 'qwen_cot',
             'qwen_few_shot_stratified']:
    p = RESULT_DIR / f'classified_{name}.json'
    if p.exists():
        runs[name] = json.load(open(p))
        print(f'  {name}: {len(runs[name])} provisions')
print(f'\nTotal runs loaded: {len(runs)}')

## 5.1 Category distribution per model × strategy

In [ ]:
# Build a comparison dataframe
rows = []
for run_name, run_data in runs.items():
    counter = Counter(p['category'] for p in run_data)
    for cat, count in counter.items():
        rows.append({'run': run_name, 'category': cat, 'count': count,
                     'pct': 100 * count / len(run_data)})
df = pd.DataFrame(rows)
pivot = df.pivot_table(index='category', columns='run', values='pct', fill_value=0)

fig, ax = plt.subplots(figsize=(14, 8))
sns.heatmap(pivot, annot=True, fmt='.1f', cmap='YlGnBu', cbar_kws={'label': '% of provisions'}, ax=ax)
ax.set_title('Category distribution (%) per model-strategy run', pad=15)
plt.tight_layout()
plt.savefig('../data/results/fig_category_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

## 5.2 Cross-run agreement (Cohen's κ)

In [ ]:
from src.analysis import compare_two_runs
from itertools import combinations

kappa_matrix = pd.DataFrame(index=list(runs), columns=list(runs), dtype=float)
for a, b in combinations(runs, 2):
    cmp = compare_two_runs(runs[a], runs[b])
    if cmp['kappa'] is not None:
        kappa_matrix.loc[a, b] = cmp['kappa']
        kappa_matrix.loc[b, a] = cmp['kappa']
for r in runs:
    kappa_matrix.loc[r, r] = 1.0

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(kappa_matrix.astype(float), annot=True, fmt='.2f', cmap='RdYlGn',
            center=0, vmin=-0.2, vmax=1.0, cbar_kws={'label': "Cohen's κ"}, ax=ax)
ax.set_title("Inter-run agreement — Cohen's κ\n(1.0 = perfect, 0 = chance, <0 = worse than chance)", pad=15)
plt.tight_layout()
plt.savefig('../data/results/fig_kappa_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

## 5.3 Category × Agreement matrix (RQ2 — cross-agreement differences)

In [ ]:
# Use stratified sample if available — otherwise fall back to non-stratified
key = 'qwen_few_shot_stratified' if 'qwen_few_shot_stratified' in runs else 'qwen_few_shot'
print(f'Using: {key}')
data = runs[key]

cross = pd.DataFrame(
    [(p['agreement'], p['category']) for p in data],
    columns=['agreement', 'category']
)
matrix = pd.crosstab(cross['category'], cross['agreement'])
# reorder by overall total
matrix = matrix.loc[matrix.sum(axis=1).sort_values(ascending=False).index]

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(matrix, annot=True, fmt='d', cmap='Blues', ax=ax)
ax.set_title(f'Provision counts: Category × Agreement\n(source: {key})', pad=15)
plt.tight_layout()
plt.savefig('../data/results/fig_category_x_agreement.png', dpi=150, bbox_inches='tight')
plt.show()
display(matrix)

## 5.4 Convergence vs fragmentation signal (RQ3)

In [ ]:
from src.analysis import convergence_signal, category_matrix
mat = category_matrix(runs[key])
sig = convergence_signal(mat)

sig_df = pd.DataFrame([
    {'category': c, 'total': info['total_provisions'],
     'coef_variation': info['coef_variation'],
     'entropy_ratio': info['entropy'] / info['max_entropy'] if info['max_entropy'] else 0,
     'signal': info['signal']}
    for c, info in sig.items()
]).sort_values('total', ascending=False)
display(sig_df)

fig, ax = plt.subplots(figsize=(10, 5))
colors = ['#2ecc71' if s == 'convergent' else '#e67e22' for s in sig_df['signal']]
ax.barh(sig_df['category'], sig_df['entropy_ratio'], color=colors)
ax.axvline(0.95, ls='--', color='gray', label='high convergence threshold')
ax.set_xlabel('Entropy ratio (1.0 = perfectly even across agreements)')
ax.set_title('Convergence signal per category\n(green = converging, orange = fragmented)')
ax.invert_yaxis()
plt.tight_layout()
plt.savefig('../data/results/fig_convergence.png', dpi=150, bbox_inches='tight')
plt.show()

## 5.5 LLM-generated cross-agreement comparison (qualitative)

In [ ]:
comp = json.load(open(RESULT_DIR / 'comparison_qwen.json'))
for item in comp[:3]:
    print('═' * 80)
    print(f"  Category: {item['category']}")
    print('═' * 80)
    print(item['analysis'][:1500])
    print()

## 5.6 Validation accuracy (after manual labelling)

In [ ]:
val_path = RESULT_DIR / 'validation_report.json'
if val_path.exists():
    report = json.load(open(val_path))
    vdf = pd.DataFrame([
        {'run': k, 'n': v['n'], 'accuracy': v['accuracy'], 'macro_f1': v['macro_f1']}
        for k, v in report.items()
    ]).sort_values('macro_f1', ascending=False)
    display(vdf)
else:
    print('Run validation first:\n'
          '  1. python -m src.validation --sample\n'
          '  2. Open data/results/validation_set.csv, fill gold_category column\n'
          '  3. python -m src.validation --evaluate')